In [42]:
# Setup, imports
import math
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib
import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import seaborn as sns
sns.set(style="white")
sns.set(style="whitegrid", color_codes=True)
from patsy import dmatrices
import statsmodels.api as sm 
import seaborn as sn
from scipy.stats import wilcoxon, pearsonr, mannwhitneyu
import scipy.stats as stats
from tableone import TableOne
from IPython.display import display

pd.set_option('display.max_rows', 200)
matplotlib.rcParams['figure.figsize'] = [40, 20]
import warnings
warnings.filterwarnings('ignore')

In [74]:
#organizing input dataframes for baseline, 3 months, 6 months followup

data = pd.read_excel('data_new_raw.xlsx')
colsBL = ['Age', 'Race', 'Indigenous', 'Gender Identity', 'Highschool Education', 'Post- Secondary Education', 'Employment', 'Household income before taxes',
            'Marital status', 'Living situation','Pack years','Alcohol use - average number of drinks/week', 'Substance Use Disorder', 'Depression',
          'Anxiety', 'Health Anxiety', 'Delay to Hospital', 'Personal Stressors', 'Menopause', 'Complicated Pregnancy', 'Stroke History', 'MI History',
          'CABG/Stent History', 'High cholesterol', 'Diabetes', 'Hypertension', 'BMI', 'HFrEF', 'STEMI', 'ESSI score']
colsOutcomesBL = ['HADS-A','HADS-D']
colsOutcomes3m = ['3M New Depression Diagnosis ', '3M New Anxiety Diagnosis ', '3M New Substance Use Disorder', '3M New Stroke', '3M New MI',
                  '3M New CABG/Stent', '3M Cardiac Rehab', '3M Hospitalized ', '3M HADS-A', '3M HADS-D', '3M SF-12 (Low)', '3M Cardiac Anxiety',
                  '3M MOS (Low Support)']
colsOutcomes6m = ['6M New Depression Diagnosis ', '6M New Anxiety Diagnosis', '6M New Substance Use Disorder', '6M New Stroke', '6M New MI',
                  '6M New CABG/Stent', '6M Cardiac Rehab', '6M Hospitalized ', '6M HADS-A', '6M HADS-D', '6M SF-12 (Low)', '6M Cardiac Anxiety',
                  '6M MOS (Low Support)']

cols3m = colsBL + colsOutcomesBL
cols6m = cols3m + colsOutcomes3m

dataBL = data[colsBL+colsOutcomesBL].copy().dropna(axis=0)
data3m = data[colsBL+colsOutcomesBL+colsOutcomes3m].copy().dropna(axis=0)
data6m = data[colsBL+colsOutcomesBL+colsOutcomes3m+colsOutcomes6m].copy().dropna(axis=0)

#defining conitnuous and categorical columns
colsCont = ['Age', 'Pack years', 'Alcohol use - average number of drinks/week','BMI']

colsCat  = colsBL.copy()
for item in colsCont:
    colsCat.remove(item)

cols3mCat = cols3m.copy()
for item in colsCont:
    cols3mCat.remove(item)
    
cols6mCat = cols6m.copy()
for item in colsCont:
    cols6mCat.remove(item)


# Computing significance tests

In [106]:
# computing table 1 stats and p values for each variable for each outcome

for outcome in colsOutcomesBL:
    print('Baseline, Outcome = ' + outcome)
    table = TableOne(dataBL, columns=colsBL, pval = True, 
                       groupby = [outcome], htest_name=True, smd=True)
    filename = outcome + "_baseline_table1.csv"
    #removing backslash from strings
    filename = filename.replace("/","-")
    table.to_csv(filename, index=False)
    
    display(table1BL)


Baseline, Outcome = HADS-A


Grouped by HADS-D                                                                            
                                                                     Missing      Overall            0            1 P-Value               Test SMD (0,1)
n                                                                                     698          570          128                                     
Age, mean (SD)                                                             0  67.0 (12.4)  67.3 (12.4)  65.7 (12.2)   0.202  Two Sample T-test    -0.125
Race, n (%)                                            0                   0   627 (89.8)   513 (90.0)   114 (89.1)   0.877        Chi-squared     0.031
                                                       1                        71 (10.2)    57 (10.0)    14 (10.9)                                     
Indigenous, n (%)                                      0                   0   673 (96.4)   549 (96.3)   124 (96.9)   1.000     Fisher's exact     0.031
                                                       1                         25 (3.6)     21 (3.7)      4 (3.1)                                     
Gender Identity, n (%)                                 0                   0   697 (99.9)   569 (99.8)  128 (100.0)   1.000     Fisher's exact       nan
                                                       1                          1 (0.1)      1 (0.2)                                                  
Highschool Education, n (%)                            0                   0   617 (88.4)   504 (88.4)   113 (88.3)   1.000        Chi-squared     0.004
                                                       1                        81 (11.6)    66 (11.6)    15 (11.7)                                     
Post- Secondary Education, n (%)                       0                   0   391 (56.0)   323 (56.7)    68 (53.1)   0.528        Chi-squared     0.071
                                                       1                       307 (44.0)   247 (43.3)    60 (46.9)                                     
Employment, n (%)                                      0                   0   611 (87.5)   508 (89.1)   103 (80.5)   0.011        Chi-squared     0.243
                                                       1                        87 (12.5)    62 (10.9)    25 (19.5)                                     
Household income before taxes, n (%)                   0.0                 0   370 (53.0)   304 (53.3)    66 (51.6)   0.791        Chi-squared     0.035
                                                       1.0                     328 (47.0)   266 (46.7)    62 (48.4)                                     
Marital status, n (%)                                  0                   0   351 (50.3)   297 (52.1)    54 (42.2)   0.054        Chi-squared     0.200
                                                       1                       347 (49.7)   273 (47.9)    74 (57.8)                                     
Living situation, n (%)                                0                   0   684 (98.0)   560 (98.2)   124 (96.9)   0.302     Fisher's exact     0.089
                                                       1                         14 (2.0)     10 (1.8)      4 (3.1)                                     
Pack years, mean (SD)                                                      0  26.7 (59.8)  25.4 (58.9)  32.4 (63.6)   0.257  Two Sample T-test     0.114
Alcohol use - average number of drinks/week, mean (SD)                     0    1.6 (3.6)    1.6 (3.6)    1.5 (3.7)   0.845  Two Sample T-test    -0.019
Substance Use Disorder, n (%)                          0                   0   685 (98.1)   562 (98.6)   123 (96.1)   0.071     Fisher's exact     0.156
                                                       1                         13 (1.9)      8 (1.4)      5 (3.9)                                     
Depression, n (%)                                      0                   0   47

Baseline, Outcome = HADS-D


Grouped by HADS-D                                                                            
                                                                     Missing      Overall            0            1 P-Value               Test SMD (0,1)
n                                                                                     698          570          128                                     
Age, mean (SD)                                                             0  67.0 (12.4)  67.3 (12.4)  65.7 (12.2)   0.202  Two Sample T-test    -0.125
Race, n (%)                                            0                   0   627 (89.8)   513 (90.0)   114 (89.1)   0.877        Chi-squared     0.031
                                                       1                        71 (10.2)    57 (10.0)    14 (10.9)                                     
Indigenous, n (%)                                      0                   0   673 (96.4)   549 (96.3)   124 (96.9)   1.000     Fisher's exact     0.031
                                                       1                         25 (3.6)     21 (3.7)      4 (3.1)                                     
Gender Identity, n (%)                                 0                   0   697 (99.9)   569 (99.8)  128 (100.0)   1.000     Fisher's exact       nan
                                                       1                          1 (0.1)      1 (0.2)                                                  
Highschool Education, n (%)                            0                   0   617 (88.4)   504 (88.4)   113 (88.3)   1.000        Chi-squared     0.004
                                                       1                        81 (11.6)    66 (11.6)    15 (11.7)                                     
Post- Secondary Education, n (%)                       0                   0   391 (56.0)   323 (56.7)    68 (53.1)   0.528        Chi-squared     0.071
                                                       1                       307 (44.0)   247 (43.3)    60 (46.9)                                     
Employment, n (%)                                      0                   0   611 (87.5)   508 (89.1)   103 (80.5)   0.011        Chi-squared     0.243
                                                       1                        87 (12.5)    62 (10.9)    25 (19.5)                                     
Household income before taxes, n (%)                   0.0                 0   370 (53.0)   304 (53.3)    66 (51.6)   0.791        Chi-squared     0.035
                                                       1.0                     328 (47.0)   266 (46.7)    62 (48.4)                                     
Marital status, n (%)                                  0                   0   351 (50.3)   297 (52.1)    54 (42.2)   0.054        Chi-squared     0.200
                                                       1                       347 (49.7)   273 (47.9)    74 (57.8)                                     
Living situation, n (%)                                0                   0   684 (98.0)   560 (98.2)   124 (96.9)   0.302     Fisher's exact     0.089
                                                       1                         14 (2.0)     10 (1.8)      4 (3.1)                                     
Pack years, mean (SD)                                                      0  26.7 (59.8)  25.4 (58.9)  32.4 (63.6)   0.257  Two Sample T-test     0.114
Alcohol use - average number of drinks/week, mean (SD)                     0    1.6 (3.6)    1.6 (3.6)    1.5 (3.7)   0.845  Two Sample T-test    -0.019
Substance Use Disorder, n (%)                          0                   0   685 (98.1)   562 (98.6)   123 (96.1)   0.071     Fisher's exact     0.156
                                                       1                         13 (1.9)      8 (1.4)      5 (3.9)                                     
Depression, n (%)                                      0                   0   47

In [108]:
for outcome in colsOutcomes3m:
    print('3 Months, Outcome = ' + outcome)
    table = TableOne(data3m, columns=cols3m, pval = True, 
                       groupby = [outcome], htest_name=True, smd=True)
    filename = outcome + "_3m_table1.csv"
    #removing backslash from strings
    filename = filename.replace("/","-")
    table.to_csv(filename, index=False)
    display(table)

3 Months, Outcome = 3M New Depression Diagnosis 


Grouped by 3M New Depression Diagnosis                                                                                 
                                                                                           Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                           463          428           35                                         
Age, mean (SD)                                                                                   0  67.4 (12.3)  67.4 (12.5)  66.6 (10.7)   0.651  Two Sample T-test        -0.075
Race, n (%)                                            0                                         0   426 (92.0)   393 (91.8)    33 (94.3)   1.000     Fisher's exact         0.097
                                                       1                                               37 (8.0)     35 (8.2)      2 (5.7)                                         
Indigenous, n (%)                                      0                                         0   450 (97.2)   416 (97.2)    34 (97.1)   1.000     Fisher's exact         0.003
                                                       1                                               13 (2.8)     12 (2.8)      1 (2.9)                                         
Gender Identity, n (%)                                 0                                         0   462 (99.8)   427 (99.8)   35 (100.0)   1.000     Fisher's exact           nan
                                                       1                                                1 (0.2)      1 (0.2)                                                      
Highschool Education, n (%)                            0                                         0   422 (91.1)   393 (91.8)    29 (82.9)   0.111     Fisher's exact         0.272
                                                       1                                               41 (8.9)     35 (8.2)     6 (17.1)                                         
Post- Secondary Education, n (%)                       0                                         0   277 (59.8)   257 (60.0)    20 (57.1)   0.875        Chi-squared         0.059
                                                       1                                             186 (40.2)   171 (40.0)    15 (42.9)                                         
Employment, n (%)                                      0                                         0   411 (88.8)   383 (89.5)    28 (80.0)   0.096     Fisher's exact         0.266
                                                       1                                              52 (11.2)    45 (10.5)     7 (20.0)                                         
Household income before taxes, n (%)                   0.0                                       0   254 (54.9)   240 (56.1)    14 (40.0)   0.097        Chi-squared         0.326
                                                       1.0                                           209 (45.1)   188 (43.9)    21 (60.0)                                         
Marital status, n (%)                                  0                                         0   239 (51.6)   225 (52.6)    14 (40.0)   0.210        Chi-squared         0.254
                                                       1                                             224 (48.4)   203 (47.4)    21 (60.0)                                         
Living situation, n (%)                                0                                         0   457 (98.7)   423 (98.8)    34 (97.1)   0.378     Fisher's exact         0.120
                                                       1                                                6 (1.3)      5 (1.2)      1 (2.9)                                         
Pack years, mean (SD)                                                                            0  31.7 (71.0)  32.9 (73

3 Months, Outcome = 3M New Anxiety Diagnosis 


Grouped by 3M New Anxiety Diagnosis                                                                                 
                                                                                        Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                        463          425           38                                         
Age, mean (SD)                                                                                0  67.4 (12.3)  67.4 (12.5)  67.0 (10.3)   0.806  Two Sample T-test        -0.038
Race, n (%)                                            0                                      0   426 (92.0)   391 (92.0)    35 (92.1)   1.000     Fisher's exact         0.004
                                                       1                                            37 (8.0)     34 (8.0)      3 (7.9)                                         
Indigenous, n (%)                                      0                                      0   450 (97.2)   414 (97.4)    36 (94.7)   0.290     Fisher's exact         0.138
                                                       1                                            13 (2.8)     11 (2.6)      2 (5.3)                                         
Gender Identity, n (%)                                 0                                      0   462 (99.8)   424 (99.8)   38 (100.0)   1.000     Fisher's exact           nan
                                                       1                                             1 (0.2)      1 (0.2)                                                      
Highschool Education, n (%)                            0                                      0   422 (91.1)   387 (91.1)    35 (92.1)   1.000     Fisher's exact         0.038
                                                       1                                            41 (8.9)     38 (8.9)      3 (7.9)                                         
Post- Secondary Education, n (%)                       0                                      0   277 (59.8)   251 (59.1)    26 (68.4)   0.339        Chi-squared         0.196
                                                       1                                          186 (40.2)   174 (40.9)    12 (31.6)                                         
Employment, n (%)                                      0                                      0   411 (88.8)   379 (89.2)    32 (84.2)   0.417     Fisher's exact         0.147
                                                       1                                           52 (11.2)    46 (10.8)     6 (15.8)                                         
Household income before taxes, n (%)                   0.0                                    0   254 (54.9)   236 (55.5)    18 (47.4)   0.425        Chi-squared         0.164
                                                       1.0                                        209 (45.1)   189 (44.5)    20 (52.6)                                         
Marital status, n (%)                                  0                                      0   239 (51.6)   222 (52.2)    17 (44.7)   0.474        Chi-squared         0.150
                                                       1                                          224 (48.4)   203 (47.8)    21 (55.3)                                         
Living situation, n (%)                                0                                      0   457 (98.7)   420 (98.8)    37 (97.4)   0.404     Fisher's exact         0.107
                                                       1                                             6 (1.3)      5 (1.2)      1 (2.6)                                         
Pack years, mean (SD)                                                                         0  31.7 (71.0)  31.7 (71.8)  32.1 (62.3)   0.967  Two Sample T-test         0.007
Alcohol use

3 Months, Outcome = 3M New Substance Use Disorder


...

Grouped by 3M SF-12 (Low)                                                                                
                                                                             Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                             463          224          239                                         
Age, mean (SD)                                                                     0  67.4 (12.3)  67.8 (12.4)  67.0 (12.3)   0.523  Two Sample T-test        -0.059
Race, n (%)                                            0                           0   426 (92.0)   215 (96.0)   211 (88.3)   0.004        Chi-squared         0.289
                                                       1                                 37 (8.0)      9 (4.0)    28 (11.7)                                         
Indigenous, n (%)                                      0                           0   450 (97.2)   218 (97.3)   232 (97.1)   1.000        Chi-squared         0.015
                                                       1                                 13 (2.8)      6 (2.7)      7 (2.9)                                         
Gender Identity, n (%)                                 0                           0   462 (99.8)  224 (100.0)   238 (99.6)   1.000     Fisher's exact           nan
                                                       1                                  1 (0.2)                   1 (0.4)                                         
Highschool Education, n (%)                            0                           0   422 (91.1)   204 (91.1)   218 (91.2)   1.000        Chi-squared         0.005
                                                       1                                 41 (8.9)     20 (8.9)     21 (8.8)                                         
Post- Secondary Education, n (%)                       0                           0   277 (59.8)   135 (60.3)   142 (59.4)   0.926        Chi-squared         0.017
                                                       1                               186 (40.2)    89 (39.7)    97 (40.6)                                         
Employment, n (%)                                      0                           0   411 (88.8)   199 (88.8)   212 (88.7)   1.000        Chi-squared         0.004
                                                       1                                52 (11.2)    25 (11.2)    27 (11.3)                                         
Household income before taxes, n (%)                   0.0                         0   254 (54.9)   128 (57.1)   126 (52.7)   0.388        Chi-squared         0.089
                                                       1.0                             209 (45.1)    96 (42.9)   113 (47.3)                                         
Marital status, n (%)                                  0                           0   239 (51.6)   122 (54.5)   117 (49.0)   0.275        Chi-squared         0.110
                                                       1                               224 (48.4)   102 (45.5)   122 (51.0)                                         
Living situation, n (%)                                0                           0   457 (98.7)   221 (98.7)   236 (98.7)   1.000     Fisher's exact         0.007
                                                       1                                  6 (1.3)      3 (1.3)      3 (1.3)                                         
Pack years, mean (SD)                                                              0  31.7 (71.0)  46.4 (92.5)  17.9 (37.1)  <0.001  Two Sample T-test        -0.405
Alcohol use - average number of drinks/week, mean (SD)                             0    1.6 (3.6)    1.5 (3.3)    1.8 (3.8)   0.339  Two Sample T-test         0.089
Substance Use Disorder, n (%)                          0                           0   458 (98.9)  

3 Months, Outcome = 3M Cardiac Anxiety


Grouped by 3M Cardiac Anxiety                                                                                
                                                                                 Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                 463          396           67                                         
Age, mean (SD)                                                                         0  67.4 (12.3)  68.0 (12.1)  63.5 (12.9)   0.008  Two Sample T-test        -0.366
Race, n (%)                                            0                               0   426 (92.0)   367 (92.7)    59 (88.1)   0.296        Chi-squared         0.157
                                                       1                                     37 (8.0)     29 (7.3)     8 (11.9)                                         
Indigenous, n (%)                                      0                               0   450 (97.2)   385 (97.2)    65 (97.0)   1.000     Fisher's exact         0.012
                                                       1                                     13 (2.8)     11 (2.8)      2 (3.0)                                         
Gender Identity, n (%)                                 0                               0   462 (99.8)   395 (99.7)   67 (100.0)   1.000     Fisher's exact           nan
                                                       1                                      1 (0.2)      1 (0.3)                                                      
Highschool Education, n (%)                            0                               0   422 (91.1)   361 (91.2)    61 (91.0)   1.000        Chi-squared         0.004
                                                       1                                     41 (8.9)     35 (8.8)      6 (9.0)                                         
Post- Secondary Education, n (%)                       0                               0   277 (59.8)   236 (59.6)    41 (61.2)   0.911        Chi-squared         0.033
                                                       1                                   186 (40.2)   160 (40.4)    26 (38.8)                                         
Employment, n (%)                                      0                               0   411 (88.8)   352 (88.9)    59 (88.1)   1.000        Chi-squared         0.026
                                                       1                                    52 (11.2)    44 (11.1)     8 (11.9)                                         
Household income before taxes, n (%)                   0.0                             0   254 (54.9)   222 (56.1)    32 (47.8)   0.259        Chi-squared         0.167
                                                       1.0                                 209 (45.1)   174 (43.9)    35 (52.2)                                         
Marital status, n (%)                                  0                               0   239 (51.6)   203 (51.3)    36 (53.7)   0.809        Chi-squared         0.049
                                                       1                                   224 (48.4)   193 (48.7)    31 (46.3)                                         
Living situation, n (%)                                0                               0   457 (98.7)   390 (98.5)   67 (100.0)   0.600     Fisher's exact           nan
                                                       1                                      6 (1.3)      6 (1.5)                                                      
Pack years, mean (SD)                                                                  0  31.7 (71.0)  32.7 (73.6)  25.7 (53.2)   0.350  Two Sample T-test        -0.109
Alcohol use - average number of drinks/week, mean (SD)                                 0    1.6 (3.6)    1.6 (3.6)    1.7 (3.6)   0.918  Two Sample T-test         0.014
Sub

3 Months, Outcome = 3M MOS (Low Support)


Grouped by 3M MOS (Low Support)                                                                                 
                                                                                   Missing      Overall          0.0           1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                   463          354           109                                         
Age, mean (SD)                                                                           0  67.4 (12.3)  67.6 (12.5)   66.7 (11.7)   0.509  Two Sample T-test        -0.071
Race, n (%)                                            0                                 0   426 (92.0)   323 (91.2)    103 (94.5)   0.372        Chi-squared         0.127
                                                       1                                       37 (8.0)     31 (8.8)       6 (5.5)                                         
Indigenous, n (%)                                      0                                 0   450 (97.2)   343 (96.9)    107 (98.2)   0.742     Fisher's exact         0.082
                                                       1                                       13 (2.8)     11 (3.1)       2 (1.8)                                         
Gender Identity, n (%)                                 0                                 0   462 (99.8)   353 (99.7)   109 (100.0)   1.000     Fisher's exact           nan
                                                       1                                        1 (0.2)      1 (0.3)                                                       
Highschool Education, n (%)                            0                                 0   422 (91.1)   321 (90.7)    101 (92.7)   0.657        Chi-squared         0.072
                                                       1                                       41 (8.9)     33 (9.3)       8 (7.3)                                         
Post- Secondary Education, n (%)                       0                                 0   277 (59.8)   218 (61.6)     59 (54.1)   0.202        Chi-squared         0.151
                                                       1                                     186 (40.2)   136 (38.4)     50 (45.9)                                         
Employment, n (%)                                      0                                 0   411 (88.8)   319 (90.1)     92 (84.4)   0.140        Chi-squared         0.172
                                                       1                                      52 (11.2)     35 (9.9)     17 (15.6)                                         
Household income before taxes, n (%)                   0.0                               0   254 (54.9)   210 (59.3)     44 (40.4)   0.001        Chi-squared         0.386
                                                       1.0                                   209 (45.1)   144 (40.7)     65 (59.6)                                         
Marital status, n (%)                                  0                                 0   239 (51.6)   190 (53.7)     49 (45.0)   0.138        Chi-squared         0.175
                                                       1                                     224 (48.4)   164 (46.3)     60 (55.0)                                         
Living situation, n (%)                                0                                 0   457 (98.7)   353 (99.7)    104 (95.4)   0.003     Fisher's exact         0.282
                                                       1                                        6 (1.3)      1 (0.3)       5 (4.6)                                         
Pack years, mean (SD)                                                                    0  31.7 (71.0)  11.9 (17.8)  95.9 (122.8)  <0.001  Two Sample T-test         0.958
Alcohol use - average number of drinks/week, mean (SD)                                   0    1.6 (3.6)

In [109]:
for outcome in colsOutcomes6m:
    print('6 Months, Outcome = ' + outcome)
    table = TableOne(data = data6m, columns=cols6m, pval = True, categorical = cols6mCat,
                       groupby = [outcome], htest_name=True, smd=True)
    filename = outcome + "_6m_table1.csv"
    #removing backslash from strings
    filename = filename.replace("/","-")
    table.to_csv(filename, index=False)
    display(table)

6 Months, Outcome = 6M New Depression Diagnosis 


Grouped by 6M New Depression Diagnosis                                                                                 
                                                                                           Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                           363          333           30                                         
Age, mean (SD)                                                                                   0  67.7 (12.5)  67.5 (12.5)  69.6 (12.0)   0.364  Two Sample T-test         0.172
Race, n (%)                                            0                                         0   337 (92.8)   310 (93.1)    27 (90.0)   0.463     Fisher's exact         0.111
                                                       1                                               26 (7.2)     23 (6.9)     3 (10.0)                                         
Indigenous, n (%)                                      0                                         0   352 (97.0)   324 (97.3)    28 (93.3)   0.228     Fisher's exact         0.188
                                                       1                                               11 (3.0)      9 (2.7)      2 (6.7)                                         
Gender Identity, n (%)                                 0                                         0   362 (99.7)   332 (99.7)   30 (100.0)   1.000     Fisher's exact           nan
                                                       1                                                1 (0.3)      1 (0.3)                                                      
Highschool Education, n (%)                            0                                         0   330 (90.9)   305 (91.6)    25 (83.3)   0.173     Fisher's exact         0.251
                                                       1                                               33 (9.1)     28 (8.4)     5 (16.7)                                         
Post- Secondary Education, n (%)                       0                                         0   221 (60.9)   204 (61.3)    17 (56.7)   0.765        Chi-squared         0.094
                                                       1                                             142 (39.1)   129 (38.7)    13 (43.3)                                         
Employment, n (%)                                      0                                         0   329 (90.6)   302 (90.7)    27 (90.0)   0.752     Fisher's exact         0.023
                                                       1                                               34 (9.4)     31 (9.3)     3 (10.0)                                         
Household income before taxes, n (%)                   0.0                                       0   203 (55.9)   187 (56.2)    16 (53.3)   0.915        Chi-squared         0.057
                                                       1.0                                           160 (44.1)   146 (43.8)    14 (46.7)                                         
Marital status, n (%)                                  0                                         0   180 (49.6)   168 (50.5)    12 (40.0)   0.365        Chi-squared         0.211
                                                       1                                             183 (50.4)   165 (49.5)    18 (60.0)                                         
Living situation, n (%)                                0                                         0   359 (98.9)   330 (99.1)    29 (96.7)   0.293     Fisher's exact         0.170
                                                       1                                                4 (1.1)      3 (0.9)      1 (3.3)                                         
Pack years, mean (SD)                                                                            0  29.6 (65.4)  31.0 (68

6 Months, Outcome = 6M New Anxiety Diagnosis


Grouped by 6M New Anxiety Diagnosis                                                                                
                                                                                       Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                       363          329           34                                         
Age, mean (SD)                                                                               0  67.7 (12.5)  67.9 (12.5)  66.1 (12.5)   0.442  Two Sample T-test        -0.140
Race, n (%)                                            0                                     0   337 (92.8)   308 (93.6)    29 (85.3)   0.083     Fisher's exact         0.274
                                                       1                                           26 (7.2)     21 (6.4)     5 (14.7)                                         
Indigenous, n (%)                                      0                                     0   352 (97.0)   321 (97.6)    31 (91.2)   0.074     Fisher's exact         0.280
                                                       1                                           11 (3.0)      8 (2.4)      3 (8.8)                                         
Gender Identity, n (%)                                 0                                     0   362 (99.7)   328 (99.7)   34 (100.0)   1.000     Fisher's exact           nan
                                                       1                                            1 (0.3)      1 (0.3)                                                      
Highschool Education, n (%)                            0                                     0   330 (90.9)   298 (90.6)    32 (94.1)   0.754     Fisher's exact         0.133
                                                       1                                           33 (9.1)     31 (9.4)      2 (5.9)                                         
Post- Secondary Education, n (%)                       0                                     0   221 (60.9)   197 (59.9)    24 (70.6)   0.301        Chi-squared         0.226
                                                       1                                         142 (39.1)   132 (40.1)    10 (29.4)                                         
Employment, n (%)                                      0                                     0   329 (90.6)   299 (90.9)    30 (88.2)   0.543     Fisher's exact         0.087
                                                       1                                           34 (9.4)     30 (9.1)     4 (11.8)                                         
Household income before taxes, n (%)                   0.0                                   0   203 (55.9)   185 (56.2)    18 (52.9)   0.852        Chi-squared         0.066
                                                       1.0                                       160 (44.1)   144 (43.8)    16 (47.1)                                         
Marital status, n (%)                                  0                                     0   180 (49.6)   164 (49.8)    16 (47.1)   0.897        Chi-squared         0.056
                                                       1                                         183 (50.4)   165 (50.2)    18 (52.9)                                         
Living situation, n (%)                                0                                     0   359 (98.9)   326 (99.1)    33 (97.1)   0.326     Fisher's exact         0.148
                                                       1                                            4 (1.1)      3 (0.9)      1 (2.9)                                         
Pack years, mean (SD)                                                                        0  29.6 (65.4)  30.1 (67.9)  24.1 (33.8)   0.386  Two Sample T-test        -0.112
Alcohol use - average number of dr

6 Months, Outcome = 6M New Substance Use Disorder


...

Grouped by 6M SF-12 (Low)                                                                                
                                                                             Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                             363          180          183                                         
Age, mean (SD)                                                                     0  67.7 (12.5)  68.1 (12.3)  67.3 (12.7)   0.504  Two Sample T-test        -0.070
Race, n (%)                                            0                           0   337 (92.8)   175 (97.2)   162 (88.5)   0.003        Chi-squared         0.343
                                                       1                                 26 (7.2)      5 (2.8)    21 (11.5)                                         
Indigenous, n (%)                                      0                           0   352 (97.0)   176 (97.8)   176 (96.2)   0.559        Chi-squared         0.094
                                                       1                                 11 (3.0)      4 (2.2)      7 (3.8)                                         
Gender Identity, n (%)                                 0                           0   362 (99.7)  180 (100.0)   182 (99.5)   1.000     Fisher's exact           nan
                                                       1                                  1 (0.3)                   1 (0.5)                                         
Highschool Education, n (%)                            0                           0   330 (90.9)   165 (91.7)   165 (90.2)   0.752        Chi-squared         0.052
                                                       1                                 33 (9.1)     15 (8.3)     18 (9.8)                                         
Post- Secondary Education, n (%)                       0                           0   221 (60.9)   107 (59.4)   114 (62.3)   0.654        Chi-squared         0.058
                                                       1                               142 (39.1)    73 (40.6)    69 (37.7)                                         
Employment, n (%)                                      0                           0   329 (90.6)   163 (90.6)   166 (90.7)   1.000        Chi-squared         0.005
                                                       1                                 34 (9.4)     17 (9.4)     17 (9.3)                                         
Household income before taxes, n (%)                   0.0                         0   203 (55.9)   103 (57.2)   100 (54.6)   0.697        Chi-squared         0.052
                                                       1.0                             160 (44.1)    77 (42.8)    83 (45.4)                                         
Marital status, n (%)                                  0                           0   180 (49.6)    98 (54.4)    82 (44.8)   0.083        Chi-squared         0.194
                                                       1                               183 (50.4)    82 (45.6)   101 (55.2)                                         
Living situation, n (%)                                0                           0   359 (98.9)   179 (99.4)   180 (98.4)   0.623     Fisher's exact         0.104
                                                       1                                  4 (1.1)      1 (0.6)      3 (1.6)                                         
Pack years, mean (SD)                                                              0  29.6 (65.4)  44.0 (85.6)  15.4 (30.1)  <0.001  Two Sample T-test        -0.445
Alcohol use - average number of drinks/week, mean (SD)                             0    1.6 (3.5)    1.6 (3.4)    1.6 (3.5)   0.998  Two Sample T-test        -0.000
Substance Use Disorder, n (%)                          0                           0   361 (99.4)  

6 Months, Outcome = 6M Cardiac Anxiety


Grouped by 6M Cardiac Anxiety                                                                                
                                                                                 Missing      Overall          0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                 363          313           50                                         
Age, mean (SD)                                                                         0  67.7 (12.5)  68.6 (12.1)  61.8 (13.1)   0.001  Two Sample T-test        -0.539
Race, n (%)                                            0                               0   337 (92.8)   296 (94.6)    41 (82.0)   0.004     Fisher's exact         0.398
                                                       1                                     26 (7.2)     17 (5.4)     9 (18.0)                                         
Indigenous, n (%)                                      0                               0   352 (97.0)   306 (97.8)    46 (92.0)   0.050     Fisher's exact         0.264
                                                       1                                     11 (3.0)      7 (2.2)      4 (8.0)                                         
Gender Identity, n (%)                                 0                               0   362 (99.7)   312 (99.7)   50 (100.0)   1.000     Fisher's exact           nan
                                                       1                                      1 (0.3)      1 (0.3)                                                      
Highschool Education, n (%)                            0                               0   330 (90.9)   281 (89.8)    49 (98.0)   0.065     Fisher's exact         0.348
                                                       1                                     33 (9.1)    32 (10.2)      1 (2.0)                                         
Post- Secondary Education, n (%)                       0                               0   221 (60.9)   187 (59.7)    34 (68.0)   0.340        Chi-squared         0.172
                                                       1                                   142 (39.1)   126 (40.3)    16 (32.0)                                         
Employment, n (%)                                      0                               0   329 (90.6)   285 (91.1)    44 (88.0)   0.442     Fisher's exact         0.100
                                                       1                                     34 (9.4)     28 (8.9)     6 (12.0)                                         
Household income before taxes, n (%)                   0.0                             0   203 (55.9)   175 (55.9)    28 (56.0)   1.000        Chi-squared         0.002
                                                       1.0                                 160 (44.1)   138 (44.1)    22 (44.0)                                         
Marital status, n (%)                                  0                               0   180 (49.6)   156 (49.8)    24 (48.0)   0.929        Chi-squared         0.037
                                                       1                                   183 (50.4)   157 (50.2)    26 (52.0)                                         
Living situation, n (%)                                0                               0   359 (98.9)   309 (98.7)   50 (100.0)   1.000     Fisher's exact           nan
                                                       1                                      4 (1.1)      4 (1.3)                                                      
Pack years, mean (SD)                                                                  0  29.6 (65.4)  31.0 (69.0)  20.5 (34.9)   0.096  Two Sample T-test        -0.193
Alcohol use - average number of drinks/week, mean (SD)                                 0    1.6 (3.5)    1.7 (3.6)    1.1 (2.2)   0.143  Two Sample T-test        -0.185
Sub

6 Months, Outcome = 6M MOS (Low Support)


Grouped by 6M MOS (Low Support)                                                                                 
                                                                                   Missing      Overall           0.0          1.0 P-Value               Test SMD (0.0,1.0)
n                                                                                                   363            90          273                                         
Age, mean (SD)                                                                           0  67.7 (12.5)   67.6 (11.3)  67.7 (12.9)   0.944  Two Sample T-test         0.008
Race, n (%)                                            0                                 0   337 (92.8)     85 (94.4)   252 (92.3)   0.656        Chi-squared         0.086
                                                       1                                       26 (7.2)       5 (5.6)     21 (7.7)                                         
Indigenous, n (%)                                      0                                 0   352 (97.0)     86 (95.6)   266 (97.4)   0.476     Fisher's exact         0.102
                                                       1                                       11 (3.0)       4 (4.4)      7 (2.6)                                         
Gender Identity, n (%)                                 0                                 0   362 (99.7)    90 (100.0)   272 (99.6)   1.000     Fisher's exact           nan
                                                       1                                        1 (0.3)                    1 (0.4)                                         
Highschool Education, n (%)                            0                                 0   330 (90.9)     85 (94.4)   245 (89.7)   0.257        Chi-squared         0.175
                                                       1                                       33 (9.1)       5 (5.6)    28 (10.3)                                         
Post- Secondary Education, n (%)                       0                                 0   221 (60.9)     50 (55.6)   171 (62.6)   0.285        Chi-squared         0.144
                                                       1                                     142 (39.1)     40 (44.4)   102 (37.4)                                         
Employment, n (%)                                      0                                 0   329 (90.6)     80 (88.9)   249 (91.2)   0.655        Chi-squared         0.078
                                                       1                                       34 (9.4)     10 (11.1)     24 (8.8)                                         
Household income before taxes, n (%)                   0.0                               0   203 (55.9)     39 (43.3)   164 (60.1)   0.008        Chi-squared         0.340
                                                       1.0                                   160 (44.1)     51 (56.7)   109 (39.9)                                         
Marital status, n (%)                                  0                                 0   180 (49.6)     40 (44.4)   140 (51.3)   0.316        Chi-squared         0.137
                                                       1                                     183 (50.4)     50 (55.6)   133 (48.7)                                         
Living situation, n (%)                                0                                 0   359 (98.9)     87 (96.7)   272 (99.6)   0.049     Fisher's exact         0.222
                                                       1                                        4 (1.1)       3 (3.3)      1 (0.4)                                         
Pack years, mean (SD)                                                                    0  29.6 (65.4)  82.5 (112.4)  12.1 (18.3)  <0.001  Two Sample T-test        -0.875
Alcohol use - average number of drinks/week, mean (SD)                                   0    1.6 (3.5)

# Computing Odds ratios

In [87]:
def getOddsRatios(data, outcomes):
    
    results  = []
    for o in outcomes:
        for cols in data.columns:
            data_crosstab = pd.crosstab(data[cols],data[o])
            tp = data_crosstab.iloc[1][1]
            tn = data_crosstab.iloc[0][0]
            fp = data_crosstab.iloc[1][0]
            fn = data_crosstab.iloc[0][1]
            if(tp == 0 or tn == 0 or fp == 0 or fn == 0):
                tp = tp + 0.5
                tn = tn + 0.5
                fp = fp + 0.5
                fn = fn + 0.5

            linOR = (tp*tn)/(fp*fn)
            SE = math.sqrt(1/tp + 1/tn + 1/fp + 1/fn)
            z = math.log(linOR)/SE
            #dont take exponential fo rnow
            CILow = math.exp(math.log(linOR) - 1.96*SE)
            CIHigh = math.exp(math.log(linOR) + 1.96*SE)
            p=stats.norm.sf(abs(z))*2

            results.append([o, cols, linOR, CILow, CIHigh, p])

    resultsDF = pd.DataFrame(results, columns = ['outcome', 'variable', 'OR',
                                                 'CILow','CIHigh','p'])
    return resultsDF

In [93]:
print('Baseline')
ORBL = getOddsRatios(dataBL, colsOutcomesBL)
display(ORBL)

print('3 Months')
OR3m = getOddsRatios(data3m, colsOutcomes3m)
display(OR3m)

print('6 Months')
OR6m = getOddsRatios(data6m, colsOutcomes6m)
display(OR6m)

ORBL.to_csv('ORBL.csv', index=False)  
OR3m.to_csv('OR3m.csv', index=False)  
OR6m.to_csv('OR6m.csv', index=False)  

Baseline


,outcome,variable,OR,CILow,CIHigh,p
0,HADS-A,Age,0.555556,0.012590,2.451519e+01,7.609690e-01
1,HADS-A,Race,1.332555,0.807276,2.199621e+00,2.615406e-01
2,HADS-A,Indigenous,0.877096,0.372956,2.062707e+00,7.637455e-01
3,HADS-A,Gender Identity,0.622177,0.025249,1.533163e+01,7.716274e-01
4,HADS-A,Highschool Education,0.765201,0.461971,1.267464e+00,2.986081e-01
5,HADS-A,Post- Secondary Education,1.139949,0.833412,1.559234e+00,4.124074e-01
6,HADS-A,Employment,1.377664,0.871115,2.178770e+00,1.706903e-01
7,HADS-A,Household income before taxes,1.348819,0.987025,1.843231e+00,6.037697e-02
8,HADS-A,Marital status,1.030670,0.754831,1.407309e+00,8.492309e-01
9,HADS-A,Living situation,3.461538,1.146970,1.044687e+01,2.757264e-02


3 Months


,outcome,variable,OR,CILow,CIHigh,p
0,3M New Depression Diagnosis,Age,1.666667,0.020222,1.373648e+02,8.204702e-01
1,3M New Depression Diagnosis,Race,0.680519,0.156690,2.955559e+00,6.074671e-01
2,3M New Depression Diagnosis,Indigenous,1.019608,0.128689,8.078370e+00,9.853291e-01
3,3M New Depression Diagnosis,Gender Identity,4.014085,0.160556,1.003569e+02,3.974111e-01
4,3M New Depression Diagnosis,Highschool Education,2.323153,0.903323,5.974653e+00,8.028616e-02
...,...,...,...,...,...,...
580,3M MOS (Low Support),3M HADS-A,3.032667,1.942047,4.735763e+00,1.067024e-06
581,3M MOS (Low Support),3M HADS-D,2.714562,1.623584,4.538630e+00,1.400681e-04
582,3M MOS (Low Support),3M SF-12 (Low),0.366826,0.233587,5.760653e-01,1.329783e-05
583,3M MOS (Low Support),3M Cardiac Anxiety,0.926759,0.498878,1.721629e+00,8.097790e-01


6 Months


,outcome,variable,OR,CILow,CIHigh,p
0,6M New Depression Diagnosis,Age,0.333333,0.006614,1.680015e+01,5.827954e-01
1,6M New Depression Diagnosis,Race,1.497585,0.422342,5.310293e+00,5.317471e-01
2,6M New Depression Diagnosis,Indigenous,2.571429,0.529600,1.248536e+01,2.413818e-01
3,6M New Depression Diagnosis,Gender Identity,3.633880,0.144891,9.113820e+01,4.325154e-01
4,6M New Depression Diagnosis,Highschool Education,2.178571,0.773679,6.134553e+00,1.404277e-01
...,...,...,...,...,...,...
749,6M MOS (Low Support),6M HADS-A,0.271954,0.161347,4.583839e-01,1.016096e-06
750,6M MOS (Low Support),6M HADS-D,0.296528,0.172048,5.110706e-01,1.204166e-05
751,6M MOS (Low Support),6M SF-12 (Low),2.390829,1.455863,3.926237e+00,5.729669e-04
752,6M MOS (Low Support),6M Cardiac Anxiety,1.863636,0.839935,4.135011e+00,1.257657e-01


# Logistic Regression

## CAVEATS FOR INTERPRETATION:

Remember to do e^{coefficient} to get logreg odds ratios

Variables with high stderror likely have low variances or not enough samples for different categories)

In [99]:
def fitLogReg(data, outcomes):
    dataTemp = data

    for o in outcomes:
        data = dataTemp
        print('\n Outcome: ' + o + '\n')
        maxParams = math.floor(len(data)/10)
        #dropping outcomes
        outcomeCol = data[o]
        
        #check variance of outcome 
        #dropping outcomes columns
        data = data.drop(columns = outcomes)
        data[o] = outcomeCol

        try:
            #first logit fit
            y = data[o]
            X = data.drop(columns = [o])
            logit_model=sm.Logit(y,X)
            result=logit_model.fit()
            print(result.summary2())

            #refitting with feature selection of top N features
            print('Refitting after feature selection \n')
            toDrop = len(X.columns) - maxParams
            LRresult = (result.summary2().tables[1])
            resultDropped = LRresult.drop(LRresult['P>|z|'].nlargest(toDrop).index)
            cols = resultDropped.index

            XFiltered = data[cols]
            logit_model=sm.Logit(y,XFiltered)
            result=logit_model.fit()
            print(result.summary2())
            LRresult = (result.summary2().tables[1])
        except:
            print('Unable to converge \n')

In [100]:
print('Baseline')
fitLogReg(dataBL, colsOutcomesBL)

Baseline

 Outcome: HADS-A

         Current function value: 0.550125
         Iterations: 35
                                           Results: Logit
Model:                          Logit                        Method:                       MLE       
Dependent Variable:             HADS-A                       Pseudo R-squared:             0.149     
Date:                           2026-03-25 21:22             AIC:                          827.9747  
No. Observations:               698                          BIC:                          964.4213  
Df Model:                       29                           Log-Likelihood:               -383.99   
Df Residuals:                   668                          LL-Null:                      -451.11   
Converged:                      0.0000                       LLR p-value:                  1.7565e-15
No. Iterations:                 35.0000                      Scale:                        1.0000    
--------------------------------

In [101]:
print('3 Months')
fitLogReg(data3m, colsOutcomes3m)

3 Months

 Outcome: 3M New Depression Diagnosis 

         Current function value: 0.187169
         Iterations: 35
                                           Results: Logit
Model:                       Logit                               Method:                   MLE       
Dependent Variable:          3M New Depression Diagnosis         Pseudo R-squared:         0.301     
Date:                        2026-03-25 21:23                    AIC:                      237.3189  
No. Observations:            463                                 BIC:                      369.7262  
Df Model:                    31                                  Log-Likelihood:           -86.659   
Df Residuals:                431                                 LL-Null:                  -124.03   
Converged:                   0.0000                              LLR p-value:              1.7820e-05
No. Iterations:              35.0000                             Scale:                    1.0000    
----------

In [102]:
print('6 Months')
fitLogReg(data6m, colsOutcomes6m)

6 Months

 Outcome: 6M New Depression Diagnosis 

         Current function value: 0.119086
         Iterations: 35
                                           Results: Logit
Model:                       Logit                               Method:                   MLE       
Dependent Variable:          6M New Depression Diagnosis         Pseudo R-squared:         0.582     
Date:                        2026-03-25 21:23                    AIC:                      176.4563  
No. Observations:            363                                 BIC:                      351.7045  
Df Model:                    44                                  Log-Likelihood:           -43.228   
Df Residuals:                318                                 LL-Null:                  -103.52   
Converged:                   0.0000                              LLR p-value:              4.7089e-09
No. Iterations:              35.0000                             Scale:                    1.0000    
----------